# Harmony Multimodel Training — Kaggle
Production path: audit → train detector → evaluate → export TorchScript/ONNX → inference API → Harmony.


In [ ]:
import os, sys, json, glob, csv, random, shutil, subprocess
from pathlib import Path
print('Python', sys.version)
print('GPU check:')
os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
print('\nKaggle inputs:')
for p in Path('/kaggle/input').glob('*'): print(' -', p)


## Required input
Attach the Harmony radiograph dataset containing the `archive/train`, `archive/valid`, and `archive/test` folders. Each split must contain `_annotations.csv` with columns `filename,width,height,class,xmin,ymin,xmax,ymax`.
Supported confirmed labels: Cavity, Fillings, Implant, Impacted Tooth.


In [ ]:
from pathlib import Path
csvs=list(Path('/kaggle/input').rglob('_annotations.csv'))
print('Annotation files found:', len(csvs))
for x in csvs: print(x)
if not csvs:
    raise FileNotFoundError('Attach the radiograph dataset first. Expected _annotations.csv under /kaggle/input.')


In [ ]:
import pandas as pd
frames=[]
for f in csvs:
    d=pd.read_csv(f); d['annotation_file']=str(f); frames.append(d)
ann=pd.concat(frames,ignore_index=True)
required={'filename','width','height','class','xmin','ymin','xmax','ymax'}
assert required.issubset(ann.columns), f'Missing columns: {required-set(ann.columns)}'
print('Rows:',len(ann),' Images:',ann.filename.nunique())
print(ann['class'].value_counts())
# Repair tiny annotation boundary errors safely: clip boxes to image bounds, then drop only degenerate boxes.
for c in ['xmin','xmax']:
    ann[c]=ann[[c,'width']].apply(lambda r: min(max(float(r[c]),0.0),float(r['width'])),axis=1)
for c in ['ymin','ymax']:
    ann[c]=ann[[c,'height']].apply(lambda r: min(max(float(r[c]),0.0),float(r['height'])),axis=1)
deg=(ann.xmax<=ann.xmin)|(ann.ymax<=ann.ymin)
print('Degenerate boxes dropped after clipping:',int(deg.sum()))
ann=ann.loc[~deg].copy()
print('Usable boxes:',len(ann))


## Training
This notebook intentionally stops before full training until the attached dataset passes the audit above. The next revision will train a torchvision detector (commercial-friendly framework path), evaluate each class, save the best checkpoint, and export ONNX for Harmony deployment.


## Train production candidate detector
Uses the dataset's existing train/valid/test split. Classes: Cavity, Fillings, Implant, Impacted Tooth. Saves best validation checkpoint and evaluates on test only after training.


In [ ]:
import torch, torchvision, math, time
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import functional as TF
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES=['__background__','Cavity','Fillings','Implant','Impacted Tooth']
CLASS_TO_ID={c:i for i,c in enumerate(CLASSES)}
print('Device:',DEVICE,' Classes:',CLASSES)


In [ ]:
def find_split(name):
    matches=[p for p in Path('/kaggle/input').rglob(name) if p.is_dir() and (p/'_annotations.csv').exists()]
    if not matches: raise FileNotFoundError(f'Could not find {name}/_annotations.csv')
    return matches[0]

class DentalDetectionDataset(Dataset):
    def __init__(self, split_dir, augment=False):
        self.root=Path(split_dir); self.augment=augment
        df=pd.read_csv(self.root/'_annotations.csv')
        df=df[df['class'].isin(CLASS_TO_ID)].copy()
        for c in ['xmin','xmax']:
            df[c]=df[[c,'width']].apply(lambda r:min(max(float(r[c]),0.0),float(r['width'])),axis=1)
        for c in ['ymin','ymax']:
            df[c]=df[[c,'height']].apply(lambda r:min(max(float(r[c]),0.0),float(r['height'])),axis=1)
        df=df[(df.xmax>df.xmin)&(df.ymax>df.ymin)]
        self.groups={k:g for k,g in df.groupby('filename')}
        self.files=sorted(self.groups)
    def __len__(self): return len(self.files)
    def __getitem__(self,idx):
        fn=self.files[idx]; g=self.groups[fn]
        img=Image.open(self.root/fn).convert('RGB')
        boxes=torch.tensor(g[['xmin','ymin','xmax','ymax']].values,dtype=torch.float32)
        labels=torch.tensor([CLASS_TO_ID[x] for x in g['class']],dtype=torch.int64)
        img=TF.to_tensor(img)
        if self.augment and random.random()<0.5:
            img=TF.hflip(img); w=img.shape[-1]; boxes[:,[0,2]]=w-boxes[:,[2,0]]
        target={'boxes':boxes,'labels':labels,'image_id':torch.tensor([idx])}
        return img,target

def collate(batch): return tuple(zip(*batch))
train_ds=DentalDetectionDataset(find_split('train'),True)
valid_ds=DentalDetectionDataset(find_split('valid'))
test_ds=DentalDetectionDataset(find_split('test'))
print('Images train/valid/test:',len(train_ds),len(valid_ds),len(test_ds))
train_loader=DataLoader(train_ds,batch_size=4,shuffle=True,num_workers=2,collate_fn=collate,pin_memory=True)
valid_loader=DataLoader(valid_ds,batch_size=2,shuffle=False,num_workers=2,collate_fn=collate)
test_loader=DataLoader(test_ds,batch_size=2,shuffle=False,num_workers=2,collate_fn=collate)


In [ ]:
weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model=fasterrcnn_resnet50_fpn_v2(weights=weights)
model.roi_heads.box_predictor=FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,len(CLASSES))
model.to(DEVICE)
optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=3e-4,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=20)
scaler=torch.amp.GradScaler('cuda',enabled=DEVICE.type=='cuda')

def validation_loss(loader):
    model.train(); vals=[]
    with torch.no_grad():
        for images,targets in loader:
            images=[x.to(DEVICE) for x in images]; targets=[{k:v.to(DEVICE) for k,v in t.items()} for t in targets]
            losses=model(images,targets); vals.append(sum(losses.values()).item())
    return sum(vals)/max(1,len(vals))

best=float('inf'); patience=5; stale=0; EPOCHS=20
for epoch in range(1,EPOCHS+1):
    model.train(); running=0.0
    for images,targets in train_loader:
        images=[x.to(DEVICE) for x in images]; targets=[{k:v.to(DEVICE) for k,v in t.items()} for t in targets]
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda',enabled=DEVICE.type=='cuda'):
            losses=model(images,targets); loss=sum(losses.values())
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
        scaler.step(optimizer); scaler.update(); running+=loss.item()
    scheduler.step(); vl=validation_loss(valid_loader)
    print(f'Epoch {epoch:02d} train_loss={running/len(train_loader):.4f} val_loss={vl:.4f}')
    if vl<best:
        best=vl; stale=0
        torch.save({'model':model.state_dict(),'classes':CLASSES,'epoch':epoch,'val_loss':vl},'/kaggle/working/harmony_detector_best.pth')
    else:
        stale+=1
        if stale>=patience:
            print('Early stopping'); break
print('Best validation loss:',best)


In [ ]:
# Test-set detection metrics at IoU 0.50, confidence 0.50 (per-class precision/recall/F1).
ck=torch.load('/kaggle/working/harmony_detector_best.pth',map_location=DEVICE,weights_only=False); model.load_state_dict(ck['model']); model.eval()
def iou(a,b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1]); x2=min(a[2],b[2]); y2=min(a[3],b[3]); inter=max(0,x2-x1)*max(0,y2-y1)
    return inter/((a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter+1e-9)
stats={i:{'tp':0,'fp':0,'fn':0} for i in range(1,len(CLASSES))}
with torch.no_grad():
    for images,targets in test_loader:
        outs=model([x.to(DEVICE) for x in images])
        for out,t in zip(outs,targets):
            pb=out['boxes'].cpu().tolist(); pl=out['labels'].cpu().tolist(); ps=out['scores'].cpu().tolist()
            gb=t['boxes'].tolist(); gl=t['labels'].tolist()
            for cid in stats:
                preds=[(b,s) for b,l,s in zip(pb,pl,ps) if l==cid and s>=0.5]; preds.sort(key=lambda x:x[1],reverse=True)
                gts=[b for b,l in zip(gb,gl) if l==cid]; used=set()
                for b,s in preds:
                    cand=[(iou(b,g),j) for j,g in enumerate(gts) if j not in used]
                    if cand and max(cand)[0]>=0.5: used.add(max(cand)[1]); stats[cid]['tp']+=1
                    else: stats[cid]['fp']+=1
                stats[cid]['fn']+=len(gts)-len(used)
report={}
for cid,s in stats.items():
    p=s['tp']/(s['tp']+s['fp']+1e-9); r=s['tp']/(s['tp']+s['fn']+1e-9); f=2*p*r/(p+r+1e-9)
    report[CLASSES[cid]]={**s,'precision':p,'recall':r,'f1':f}
print(json.dumps(report,indent=2))
with open('/kaggle/working/harmony_detector_metrics.json','w') as f: json.dump(report,f,indent=2)


In [ ]:
# Export TorchScript-compatible model artifact. ONNX export can be added after metric acceptance.
model.eval().cpu()
torch.save({'model':model.state_dict(),'classes':CLASSES},'/kaggle/working/harmony_detector_final.pth')
print('OUTPUTS:')
for p in Path('/kaggle/working').glob('harmony_detector*'): print(p, round(p.stat().st_size/1024/1024,2),'MB')
